# AIWorkValue: From Experimental Evidence to a Business Decision

Independent reanalysis of the Noy–Zhang (2023) experiment with explicit commercial scenarios. This project did not conduct the original experiment or benchmark current AI products. First run `make download && make run` from the project root.

In [1]:
from pathlib import Path
import json, sys, sqlite3
ROOT = Path.cwd()
if not (ROOT / "reports/results.json").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from aiworkvalue.data import load_study
d, audit = load_study(ROOT)
r = json.loads((ROOT / "reports/results.json").read_text())
for row in audit["sample_flow"]: print(row)
print("Primary incentive arms:", int(d.primary_arm.sum()))


{'stage': 'raw_survey_records', 'n': 786, 'control': 368, 'ai_offer': 365}
{'stage': 'completed_and_not_declined', 'n': 523, 'control': 273, 'ai_offer': 250}
{'stage': 'unique_first_submission', 'n': 505, 'control': 263, 'ai_offer': 242}
{'stage': 'documented_first_task_and_duplicate_flags', 'n': 499, 'control': 260, 'ai_offer': 239}
{'stage': 'author_quality_check', 'n': 457, 'control': 237, 'ai_offer': 220}
{'stage': 'exclude_prior_graders', 'n': 453, 'control': 235, 'ai_offer': 218}
Primary incentive arms: 359


### Warehouse grain and the statistical input

The warehouse contains all retained participants, including the fixed-time arm. Outcome facts have one row per participant and measurement period. The participant mart pivots those facts before joining usage, preserving one row per person; the next primary analyses then select the appropriate incentive arms.

In [2]:
import pandas as pd
warehouse = json.loads((ROOT / "reports/warehouse-summary.json").read_text())
print("Warehouse table counts:", warehouse["table_rows"])
with sqlite3.connect(ROOT / "data/processed/study.sqlite") as connection:
    d = pd.read_sql_query("SELECT * FROM mart_participant ORDER BY prolific_pid", connection)
    print("Participant mart rows / unique IDs:", len(d), d.prolific_pid.nunique())
d["primary_arm"] = d.primary_arm.astype(bool)
assert len(d) == d.prolific_pid.nunique() == audit["retained_participants"]
print("Warehouse integrity:", warehouse["integrity_checks"])


Warehouse table counts: {'dim_participant': 453, 'dim_assignment': 2, 'dim_occupation': 6, 'dim_incentive': 3, 'dim_period': 2, 'fact_outcome': 906, 'fact_usage': 453}
Participant mart rows / unique IDs: 453 453
Warehouse integrity: [{'check_name': 'dimension_join_row_expansion', 'violations': 0}, {'check_name': 'participant_mart_grain', 'violations': 0}, {'check_name': 'missing_or_extra_measurement_period', 'violations': 0}, {'check_name': 'assignment_changes_between_periods', 'violations': 0}, {'check_name': 'usage_join_count', 'violations': 0}, {'check_name': 'outcome_grain_duplicates', 'violations': 0}]


## 1. Preserve the Assigned-Group Comparison

Actual AI use is a choice made after assignment, so it must not be used to select the treatment sample. The fixed-time incentive arm is excluded from the primary productivity comparison.

In [3]:
with sqlite3.connect(ROOT / "data/processed/study.sqlite") as connection:
    q = """SELECT assignment, COUNT(*), ROUND(AVG(time_pre),2),
    ROUND(AVG(time_post),2), ROUND(AVG(grade_post),3)
    FROM participants WHERE primary_arm=1 GROUP BY assignment"""
    print("assignment / N / baseline minutes / post minutes / post grade")
    for row in connection.execute(q): print(row)


assignment / N / baseline minutes / post minutes / post grade
(0, 187, 28.97, 27.04, 3.83)
(1, 172, 29.8, 16.93, 4.518)


### SQL: From Sample Checks to Business Scenarios

Execute the repository's SQL files directly. Paired changes are unadjusted descriptive comparisons, distinct from the regression estimates below. SQL independently calculates the commercial cases using the same assumptions and checks them against Python. Missing follow-up responses must not be treated as recorded non-use.

In [4]:
from aiworkvalue.sql_analysis import query_file
with sqlite3.connect(ROOT / "data/processed/study.sqlite") as connection:
    for name in ("02_group_coverage.sql", "03_baseline_balance.sql", "04_paired_changes.sql"):
        print("\n", name)
        for row in query_file(connection, ROOT / "sql" / name): print(row)



 02_group_coverage.sql
{'assignment': 0, 'retained_primary_arm': 187, 'time_pairs': 180, 'quality_pairs': 186, 'time_pair_coverage_pct': 96.26, 'quality_pair_coverage_pct': 99.47, 'missing_time_pairs': 7, 'missing_quality_pairs': 1}
{'assignment': 1, 'retained_primary_arm': 172, 'time_pairs': 166, 'quality_pairs': 166, 'time_pair_coverage_pct': 96.51, 'quality_pair_coverage_pct': 96.51, 'missing_time_pairs': 6, 'missing_quality_pairs': 6}

 03_baseline_balance.sql
{'outcome': 'quality_points', 'control_n': 186, 'ai_n': 166, 'control_mean': 3.9696236559139786, 'ai_mean': 3.968875502008032, 'baseline_mean_difference': -0.0007481539059464026, 'standardized_mean_difference': -0.0005852840315863421}
{'outcome': 'time_minutes', 'control_n': 180, 'ai_n': 167, 'control_mean': 28.969444444444445, 'ai_mean': 29.802395209580837, 'baseline_mean_difference': 0.8329507651363919, 'standardized_mean_difference': 0.05707506141260815}

 04_paired_changes.sql
{'outcome': 'quality_points', 'control_n': 1

In [5]:
with sqlite3.connect(ROOT / "data/processed/study.sqlite") as connection:
    for row in query_file(connection, ROOT / "sql/07_business_scenarios.sql"):
        expected = r["business"][row["scenario"]]["net_capacity_value_month"]
        assert abs(row["net_capacity_value"] - expected) < 1e-8
        print(row["scenario"], "SQL monthly net capacity value:", round(row["net_capacity_value"], 2))
sql_results = json.loads((ROOT / "reports/sql-summaries.json").read_text())
print("Python / SQL reconciliation:", sql_results["python_sql_reconciliation"])


conservative SQL monthly net capacity value: -3484.32
base SQL monthly net capacity value: 698.03
upside SQL monthly net capacity value: 4880.37
Python / SQL reconciliation: {'status': 'passed', 'business_cases': 3, 'sensitivity_points': 36, 'max_absolute_numeric_difference': 0.0}


## 2. Re-estimate a Primary Model

Adjust for the baseline outcome, occupation by task order, and incentive arm. The coefficient is measured in minutes; a negative value means faster completion. Distinguish the retained available-case sample from the full randomized population.

In [6]:
import statsmodels.formula.api as smf
primary = d[d.primary_arm].copy()
formula = "time_post ~ assignment + time_pre + C(occupation)*C(a_first) + C(incentive_arm)"
fit = smf.ols(formula, data=primary).fit(cov_type="HC3", use_t=True)
print("N:", int(fit.nobs))
print("Assignment coefficient (minutes):", round(fit.params["assignment"], 4))
print("95% interval:", fit.conf_int().loc["assignment"].round(4).tolist())
assert abs(fit.params["assignment"] - r["primary"][0]["coefficient"]) < 1e-8


N: 346
Assignment coefficient (minutes): -10.3395
95% interval: [-12.5826, -8.0964]


![Primary effects](../figures/02_primary_effects.png)

Quality is a difference in writing-rating points on a 1–7 scale, not factual accuracy.

In [7]:
for row in r["primary"]:
    print(row["name"], "N=", row["n"], "effect=", round(row["coefficient"],3),
          "CI=", [round(x,3) for x in row["ci95"]], "Holm p=", row["holm_p_value"])


primary_time N= 346 effect= -10.339 CI= [-12.583, -8.096] Holm p= 2.1972288317847033e-17
primary_quality N= 352 effect= 0.7 CI= [0.466, 0.933] Holm p= 8.987254580400753e-09


## 3. Robustness and Exploratory Differences

Different within-group significance levels do not establish a difference between groups; inspect the interaction instead. Occupation and baseline writing scores should not become employee-ability rankings.

In [8]:
for row in r["robustness"]:
    print(row["name"], "N=", row["n"], "effect=", round(row["coefficient"],3))
print("Exploratory interaction tests:")
for row in r["heterogeneity"]["interaction_tests"]:
    print(row["name"], round(row["coefficient"],3), [round(x,3) for x in row["ci95"]])


unadjusted_time N= 356 effect= -10.114
time_change_design_controls N= 346 effect= -10.842
log_time_ancova N= 346 effect= -0.657
objective_page_duration N= 357 effect= -9.32
unadjusted_quality N= 352 effect= 0.688
quality_change_design_controls N= 352 effect= 0.706
quality_including_fixed_time_arm N= 444 effect= 0.721
paired_grader_fixed_effects N= 1067 effect= 0.712
Exploratory interaction tests:
time_baseline_writing_score_interaction -0.417 [-2.131, 1.297]
quality_baseline_writing_score_interaction -0.249 [-0.434, -0.063]


## 4. Change the Business Assumptions

The estimate describes a historical assignment effect. Task frequency, transferability, verification effort and seat costs are business assumptions. Vary additional checking time to see how the commercial conclusion can reverse.

In [9]:
from aiworkvalue.business import Scenario, value_case
from dataclasses import replace
saved = -r["primary"][0]["coefficient"]
for review in (1, 2, 3, 4):
    case = value_case(saved, replace(Scenario(), additional_review_minutes=review))
    print("Additional review minutes:", review,
          "monthly net capacity value:", round(case["net_capacity_value_month"],2))
print("Released capacity is not automatically cash savings.")


Additional review minutes: 1 monthly net capacity value: 1864.69
Additional review minutes: 2 monthly net capacity value: 698.03
Additional review minutes: 3 monthly net capacity value: -468.64
Additional review minutes: 4 monthly net capacity value: -1635.31
Released capacity is not automatically cash savings.


![Business sensitivity](../figures/07_business_sensitivity.png)

## 5. Task Extension: A Weekly Project-Status Email

Compare 4, 8 and 20 eligible tasks per month with 1, 3 and 5 extra checking minutes. The hypothetical baseline is 10 minutes; transferred savings cannot exceed it. Additional revision is 0.5 minutes and expected rework is 20% × 3 minutes per task. Other costs and 50% transferability remain fixed. All seat/support costs are assigned to this workflow. This is a sensitivity exercise, not a new email experiment.

In [10]:
from aiworkvalue.task_extension import evaluate_task_extension
config = json.loads((ROOT / "data/scenarios/weekly_status_email.json").read_text())
task_sql = (ROOT / "sql/extensions/weekly_status_email.sql").read_text()
extension = evaluate_task_extension(r["primary"][0], config, task_sql)
for row in extension["cases"]:
    print("tasks/month:", row["tasks_per_employee_month"],
          "extra checking minutes:", row["review_minutes"],
          "net capacity value:", round(row["net_capacity_value"], 2))
print("SQL/Python check:", extension["reconciliation"])
assert extension == json.loads((ROOT / "reports/extensions/weekly_status_email.json").read_text())


tasks/month: 4.0 extra checking minutes: 1.0 net capacity value: -2283.73
tasks/month: 4.0 extra checking minutes: 3.0 net capacity value: -2750.39
tasks/month: 4.0 extra checking minutes: 5.0 net capacity value: -3217.06
tasks/month: 8.0 extra checking minutes: 1.0 net capacity value: -1567.46
tasks/month: 8.0 extra checking minutes: 3.0 net capacity value: -2500.79
tasks/month: 8.0 extra checking minutes: 5.0 net capacity value: -3434.12
tasks/month: 20.0 extra checking minutes: 1.0 net capacity value: 581.36
tasks/month: 20.0 extra checking minutes: 3.0 net capacity value: -1751.97
tasks/month: 20.0 extra checking minutes: 5.0 net capacity value: -4085.31
SQL/Python check: {'status': 'passed', 'cases': 9, 'max_absolute_difference': 0.0}


![Task frequency extension](../figures/extensions/weekly_status_email.png)

**Interpretation:** weekly-only use can release time without covering a full month's costs. More eligible tasks can make a low-checking scenario positive. This is not a purchase conclusion: actual task duration, task frequency, verification and rework must be measured. A very short email may not support the assumed historical saving. See `reports/extensions/Weekly_Status_Email.md` for assumptions and results.